#Pricing processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/2_utilities

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")
dbutils.widgets.text("company", "parent", "Company")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
company = dbutils.widgets.get("company")

base_path = f's3://bucket-storage-company-parents-child/{company}/{data_source}/*.csv'
print(base_path)

###Bronze

In [0]:

df= (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema_parent}.{data_source}")

###Silver

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema_parent}.{data_source};")
df_bronze.show(10)

**Transformations**

In [0]:

df_silver = df_bronze.withColumn(
    "price_inr",
    F.when(F.col("price_inr").rlike(r'^-?\d+(\.\d+)?$'), 
           F.when(F.col("price_inr").cast("double") < 0, -1 * F.col("price_inr").cast("double"))
            .otherwise(F.col("price_inr").cast("double")))
    .otherwise(0)
)

In [0]:
df_silver.show(10)


In [0]:
df_products = spark.table("atlone_enterprise.silver_parent.products")
df_leftsemi = df_silver.join(df_products, on="product_code", how="leftsemi")

In [0]:
from pyspark.sql.window import Window

# 1. Convertimos los 0 a nulos (PySpark ignora nulos al buscar)
df_silver_nulls = df_leftsemi.withColumn(
    "price_inr", 
    F.when(F.col("price_inr") == 0, None).otherwise(F.col("price_inr"))
)

# 2. Definimos las ventanas: una mira hacia atrás y otra mira hacia adelante
window_ffill = Window.partitionBy("product_code").orderBy("year").rowsBetween(Window.unboundedPreceding, Window.currentRow)
window_bfill = Window.partitionBy("product_code").orderBy("year").rowsBetween(Window.currentRow, Window.unboundedFollowing)

# 3. Aplicamos Coalesce: intenta rellenar hacia adelante (Forward Fill), si no puede, rellena hacia atrás (Backward Fill)
df_silver_final = df_silver_nulls.withColumn(
    "price_inr", 
    F.coalesce(
        F.last("price_inr", ignorenulls=True).over(window_ffill),
        F.first("price_inr", ignorenulls=True).over(window_bfill)
    )
)

In [0]:
df_silver_final.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog}.{silver_schema_parent}.{data_source}")

###Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema_parent}.{data_source};")

In [0]:
# selecionar solo las columnas requeridas
df_gold = df_silver.select("product_code", "price_inr", "year")
df_gold.show(5)

In [0]:
df_gold.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.parent_dim_{data_source}")